# Filtering pathogen-specific bioassays

## 00. Setup

In [30]:
from pathlib import Path
import shutil
import os
import json
from bs4 import BeautifulSoup
import gzip
from tqdm import tqdm
import zipfile
import pandas as pd

In [2]:
# Project paths
NOTEBOOK_DIR = Path().resolve()
DATA_RAW = NOTEBOOK_DIR.parent / "data" / "raw"
DATA_PROCESSED = NOTEBOOK_DIR.parent / "data" / "processed"
PUBCHEM_DIR = DATA_RAW / "pubchem_bioassays"
DESC_DIR = PUBCHEM_DIR / "Description"
DATA_DIR = PUBCHEM_DIR / "Data"

## 01. Prepare folders for KEPT assays

In [3]:
KEEP_DIR = DATA_RAW / "filtered_assays"
KEEP_DESC = KEEP_DIR / "Description"
KEEP_DATA = KEEP_DIR / "Data"

KEEP_DESC.mkdir(parents=True, exist_ok=True)
KEEP_DATA.mkdir(parents=True, exist_ok=True)

## 02. Load pathogen TaxIDs

In [4]:
# Load dictionary mapping pathogen → list of TaxIDs
dict_taxonomy = json.load(open(DATA_PROCESSED / "dict_taxonomy.json"))

In [15]:
# Turn dict_taxonomy into ONE set of all taxids we want
target_taxid_set = set()
for lst in dict_taxonomy.values():
    target_taxid_set.update(map(str, lst))

len(target_taxid_set), list(target_taxid_set)[:10]

(2288,
 ['32024',
  '1345500',
  '1236000',
  '2829105',
  '1055530',
  '1219276',
  '1003522178',
  '741093',
  '1002648170',
  '1416673'])

## 03. Function to extract TaxIDs from a .descr.xml

PubChem XML stores protein/gene/strain targets like:

In [ ]:
<PC-AssayDescription_target>
    <PC-AssayTarget_tax-id>1773</PC-AssayTarget_tax-id>
</PC-AssayDescription_target>

An assay may have 0, 1, or many targets, so we'll make `targets` a list of all these nodes.

In [28]:
def extract_taxids_from_descr(xml_path):
    """
    Extract all taxonomy IDs from a PubChem .descr.xml or .descr.xml.gz file.
    Returns a set of TaxID strings.
    """

    # ----- Load XML (compressed or plain) -----
    if str(xml_path).endswith(".gz"):
        with gzip.open(xml_path, "rb") as f:
            xml_content = f.read()
    else:
        with open(xml_path, "rb") as f:
            xml_content = f.read()

    soup = BeautifulSoup(xml_content, "lxml-xml")

    taxids = set()  # A set automatically removes duplicates: {"1773", "1773", "1773"} → {"1773"}

    # ----- (1) Extract taxids from target block -----
    for t in soup.find_all("PC-AssayTarget_tax-id"):
        if t.text.isdigit():
            taxids.add(t.text)

    # ----- (2) Extract taxids from XRef block -----
    for x in soup.find_all("PC-XRefData_taxonomy"):
        if x.text.isdigit():
            taxids.add(x.text)

    # ----- (3) Rare: result-type taxonomy -------
    for r in soup.find_all("PC-AssayResultType_tax-id"):
        if r.text.isdigit():
            taxids.add(r.text)

    return taxids

## 04. Loop through ALL Description files, filter and keep/delete uncompressed XML

In [16]:
# Find all ZIP files in Description folder
zip_files = list(DESC_DIR.glob("*.zip"))
len(zip_files)

1792

In [ ]:
# Storage for collected pairs
records = []     # will store dicts: {"AID": X, "TaxIDs": [...]} 

# Choose ONE ZIP file for testing
test_zip = DESC_DIR / "1368001_1369000.zip" # We chose this one since we know it contains AID 1368269 for Acinetobacter baumannii
zip_chunk = test_zip.stem
print("Testing ZIP:", test_zip)

# Temporary extraction folder
temp_extract = DESC_DIR / f"{test_zip.stem}_tmp"
temp_extract.mkdir(exist_ok=True)
print("Extracting into:", temp_extract)

# 1) Extract ZIP contents temporarily
with zipfile.ZipFile(test_zip, "r") as zf:
    zf.extractall(temp_extract)

# 2) Collect XML and XML.GZ files
xml_files = list(temp_extract.rglob("*.xml.gz")) + list(temp_extract.rglob("*.xml"))
print("Found XML files:", len(xml_files))

matched_for_test = set()

# 3) Process each XML file
for xml_path in tqdm(xml_files, desc="Scanning extracted XMLs"):
    taxids = extract_taxids_from_descr(xml_path)

    # If assay matches ANY pathogen taxid → keep it
    if taxids & target_taxid_set:
        aid = int(Path(xml_path).stem.split(".")[0])
        matched_for_test.add(aid)

        # Save record for later DATA extraction
        records.append({
            "AID": aid,
            "TaxIDs_detected": sorted(list(taxids)),
            "ZipFolder": zip_chunk
        })

        # Save uncompressed XML into KEEP_DESC
        dest_path = KEEP_DESC / f"{aid}.xml"

        if str(xml_path).endswith(".gz"):
            with gzip.open(xml_path, "rb") as f_in:
                with open(dest_path, "wb") as f_out:
                    f_out.write(f_in.read())
        else:
            shutil.copy2(xml_path, dest_path)

# Remove temporary folder
shutil.rmtree(temp_extract)
print("✓ Removed temporary extraction folder")

# Show results
print("\n✔ Test complete!")
print("Matched AIDs:", matched_for_test)
print("Number matched:", len(matched_for_test))

# Convert to DataFrame (for later use)
df_test_results = pd.DataFrame(records)
df_test_results

Testing ZIP: /Users/maria/Documents/Ersilia/PubChem/pubchem-antimicrobial-tasks/data/raw/pubchem_bioassays/Description/1368001_1369000.zip
Extracting into: /Users/maria/Documents/Ersilia/PubChem/pubchem-antimicrobial-tasks/data/raw/pubchem_bioassays/Description/1368001_1369000_tmp
Found XML files: 1000


Scanning extracted XMLs: 100%|██████████| 1000/1000 [00:44<00:00, 22.68it/s]


✓ Removed temporary extraction folder

✔ Test complete!
Matched AIDs: {1368991, 1368992, 1368993, 1368994, 1368355, 1368275, 1368746, 1368747, 1368748, 1368006, 1368268, 1368269, 1368271, 1368272, 1368273, 1368274, 1368411, 1368276, 1368277, 1368278, 1368279, 1368280, 1368281, 1368282, 1368283, 1368284, 1368412, 1368413, 1368415, 1368288, 1368289, 1368290, 1368291, 1368414, 1368285, 1368287, 1368286, 1368416, 1368417, 1368299, 1368174, 1368176, 1368702}
Number matched: 43


,AID,TaxIDs_detected,ZipFolder
0,1368355,[5833],1368001_1369000
1,1368288,[470],1368001_1369000
2,1368277,[1280],1368001_1369000
3,1368412,[1773],1368001_1369000
4,1368991,[208964],1368001_1369000
5,1368415,[1773],1368001_1369000
6,1368276,[470],1368001_1369000
7,1368289,[470],1368001_1369000
8,1368271,[1280],1368001_1369000
9,1368414,[1773],1368001_1369000


In [ ]:
def process_description_zip(zip_path, target_taxid_set):
    """
    Process ONE PubChem Description ZIP file:
    - Extract temporarily
    - Scan each XML/XML.GZ
    - If assay contains target TaxID → save uncompressed XML to KEEP_DESC
    - Remove all extracted files afterwards
    - Leave the original .zip untouched
    """

    zip_path = Path(zip_path)
    print(f"\n📦 Processing ZIP: {zip_path.name}")

    # Temporary extraction folder
    tmp_dir = zip_path.with_suffix("").with_name(zip_path.stem + "_tmp")
    tmp_dir.mkdir(exist_ok=True)
    print("→ Extracting into:", tmp_dir)

    # 1. Extract ZIP
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(tmp_dir)

    # 2. Locate XML files inside extracted folder
    xml_files = list(tmp_dir.rglob("*.xml")) + list(tmp_dir.rglob("*.xml.gz"))
    print("→ Found XML files:", len(xml_files))

    # 3. Process XML files
    matched_aids = set()

    for xml_path in tqdm(xml_files, desc="→ Scanning XML"):
        taxids = extract_taxids_from_descr(xml_path)

        if taxids & target_taxid_set:
            # This assay matches → keep uncompressed description file
            aid = int(Path(xml_path).stem.split(".")[0])
            matched_aids.add(aid)

            dest_path = KEEP_DESC / f"{aid}.xml"

            # Always save UNCOMPRESSED xml
            with gzip.open(xml_path, "rb") if str(xml_path).endswith(".gz") else open(xml_path, "rb") as f_in:
                with open(dest_path, "wb") as f_out:
                    f_out.write(f_in.read())

    # 4. Cleanup temporary folder
    shutil.rmtree(tmp_dir)
    print("✓ Removed temporary extraction folder")

    print(f"✓ Kept {len(matched_aids)} XML descriptions")
    return matched_aids